# Plot the results of the growth rate estimations

This file outputs the values of the growth rates that have the lowest average mean square error and generates Figure 3a.

Note: This assumes that there are no duplicate parameter pairs in the results files.

## Prep

Load needed packages

In [ ]:
%matplotlib widget
import matplotlib as mpl
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import math
import seaborn as sns
import os
from estimator import Estimator
pd.options.mode.chained_assignment = None

Close any figures generated from previous runs

In [ ]:
plt.close("all")

## Define data and user-input parameters

Define the file path for the configuration file (which contains the path to the growth data), save path, and names of the nude groups.

In [ ]:
groups = ["Grp. B1 nude (100% C1)", "Grp. B5 nude (100% C11)"]
result_path = "results/growth/"
config = "config_subclone.json"
save_fig_file = "figures/growth_rate_mse.svg" # Set to None if you don't want to save the figure

Create and Estimator object from the config file

In [ ]:
es = Estimator(config)

## Load data

Loop through the data files and create a data frame for each subclone with all the mean squared error results.

In [ ]:
results_c1 = []
results_c11 = []
for fname in [f for f in os.listdir(result_path) if ".csv" in f]:
    if "C1_" in fname:
        results_c1 += [pd.read_csv("{}/{}".format(result_path, fname))]
    else:
        results_c11 += [pd.read_csv("{}/{}".format(result_path, fname))]
results_c1 = pd.concat(results_c1)
results_c11 = pd.concat(results_c11)

In [ ]:
results_c1

In [ ]:
results_c11

## Determine optimal growth rate values

In [ ]:
avgs_c1 = results_c1.groupby("g")["error"].mean().reset_index()
min_c1 = avgs_c1[avgs_c1["error"] == avgs_c1["error"].min()]
print("C1: g={}, error={}".format(round(min_c1["g"].to_list()[0], 3), round(min_c1["error"].to_list()[0], 3)))
avgs_c11 = results_c11.groupby("g")["error"].mean().reset_index()
min_c11 = avgs_c11[avgs_c11["error"] == avgs_c11["error"].min()]
print("C11: g={}, error={}".format(round(min_c11["g"].to_list()[0], 3), round(min_c11["error"].to_list()[0], 3)))

## Plot average MSE for each subclone

Subset the data into the points in the top 50th percentile for each mouse in order to make the range of values to plot such that the gradient is visible. Then take the average of the MSE across all mice.

In [ ]:
# C1
# List to contain the data updated for the 50th quantile cut off for C1 mice
cut_avg_c1 = []
# Loop through the number of mice in C1
for i in range(len(results_c1["id"].unique())):
    # Pull out the mouse id
    mid = results_c1["id"].unique()[i]
    # Extract the data for that mouse id
    tmp = results_c1[results_c1["id"] == mid]
    # Find the top 50th percentile
    q = np.quantile(tmp["error"], 0.5)
    # Set MSE values larger than the 50th percentile to the 50th percentile
    tmp["error_cut"] = tmp["error"].where(tmp["error"] <= q, q)
    # Add the updated data frame to the list
    cut_avg_c1 += [tmp.reset_index(drop=True)]
# Convert the list of data frames into a single data frame
cut_avg_c1 = pd.concat(cut_avg_c1)
# Average across growth rate values
cut_avg_c1 = cut_avg_c1.groupby("g")["error_cut"].mean().reset_index()
# Set index to the growth rate values
cut_avg_c1.index = cut_avg_c1["g"]
# Sort
cut_avg_c1 = cut_avg_c1.sort_index(ascending=False)

# C11
# List to contain the data with updated for the 50th quantile cut off for C11 mice
cut_avg_c11 = []
# Loop through the number of mice in C11
for i in range(len(results_c11["id"].unique())):
    # Pull out the mouse id
    mid = results_c11["id"].unique()[i]
    # Extract the data for that mouse id
    tmp = results_c11[results_c11["id"] == mid]
    # Find the top 50th percentile
    q = np.quantile(tmp["error"], 0.5)
    # Set MSE values larger than the 50th percentile to the 50th percentile
    tmp["error_cut"] = tmp["error"].where(tmp["error"] <= q, q)
    # Add the updated data frame to the list
    cut_avg_c11 += [tmp.reset_index(drop=True)]
# Convert the list of data frames into a single data frame
cut_avg_c11 = pd.concat(cut_avg_c11)
# Average across growth rate values
cut_avg_c11 = cut_avg_c11.groupby("g")["error_cut"].mean().reset_index()
# Set index to the growth rate values
cut_avg_c11.index = cut_avg_c11["g"]
# Sort
cut_avg_c11 = cut_avg_c11.sort_index(ascending=False)

Plot the average MSE of each growth rate for both C1 and C11. Save if save_fig_file is specified.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2)

# Plot C1
sns.heatmap(cut_avg_c1["error_cut"].to_numpy().reshape(-1, 1), yticklabels=[round(j, 3) for j in cut_avg_c1.index], ax=axes[0], cmap="hot")
axes[0].set_yticks(axes[0].get_yticks()[9:][::10])
axes[0].set_title("Avg C1 growth rates")

# Plot C11
sns.heatmap(cut_avg_c11["error_cut"].to_numpy().reshape(-1, 1), yticklabels=[round(j, 3) for j in cut_avg_c11.index], ax=axes[1], cmap="hot")
axes[1].set_yticks(axes[1].get_yticks()[9:][::10])
axes[1].set_title("Avg C11 growth rates")

plt.tight_layout()
if save_fig_file:
    plt.savefig(save_fig_file)
plt.show()